# Spider: twenty questions first

The benchmark graph (`backend/eval/graph.py`) on 20 random Spider dev questions, before spending on all 1,034.

What this is for:
- **Find our bugs.** The first failures are usually in the harness, not the model.
- **Measure real tokens and the repair rate**, so the full-run cost estimate stops being a guess.
- **Write `pred_20.sql` and `gold_20.sql`** for the official scorer (block 5).

**Cost:** about 20–25 LLM calls, roughly $0.005. Nothing runs until you run the cells below.

In [1]:
# Jupyter starts in notebooks/, so put the repo root on sys.path for `backend`
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import asyncio, json, random, time
from collections import Counter

from langchain_core.callbacks import UsageMetadataCallbackHandler

from backend.eval.graph import build_graph, execute_sql
from backend.graph.build import initial_state

SPIDER = ROOT / "data" / "spider"
OUT = ROOT / "data" / "eval"
OUT.mkdir(parents=True, exist_ok=True)

def db_path(db_id):
    return str(SPIDER / "database" / db_id / f"{db_id}.sqlite")

graph = build_graph()
print(graph.get_graph().draw_ascii())

                                +-----------+                                 
                                | __start__ |                                 
                                +-----------+                                 
                                       *                                      
                                       *                                      
                                       *                                      
                                +------------+                                
                                | understand |                                
                                +------------+                                
                                       *                                      
                                       *                                      
                                       *                                      
                             +-----------------+    

## Pick 20

The dev file is sorted by database, so the first 20 would all be `concert_singer`. A fixed random sample spreads them across databases, and the seed keeps it the same every run.

In [2]:
dev = json.load(open(SPIDER / "dev.json", encoding="utf-8"))
sample = random.Random(0).sample(dev, 20)
print(Counter(q["db_id"] for q in sample).most_common())

# the official scorer wants gold lines as "SQL<TAB>db_id", in the same order as the predictions
(OUT / "gold_20.sql").write_text("".join(f"{q['query']}\t{q['db_id']}\n" for q in sample), encoding="utf-8")

[('student_transcripts_tracking', 3), ('world_1', 2), ('orchestra', 2), ('dog_kennels', 2), ('tvshow', 2), ('employee_hire_evaluation', 2), ('flight_2', 2), ('pets_1', 1), ('wta_1', 1), ('real_estate_properties', 1), ('cre_Doc_Template_Mgt', 1), ('car_1', 1)]


2330

## Run them

Five at a time (`Semaphore(5)`), each with its own token counter. The counter is a LangChain callback: it sees every LLM call inside the graph without the nodes knowing it's there.

In [3]:
sem = asyncio.Semaphore(5)

async def run_one(q):
    usage = UsageMetadataCallbackHandler()
    async with sem:
        started = time.monotonic()
        state = await graph.ainvoke(
            initial_state(q["question"]),
            config={"configurable": {"db_path": db_path(q["db_id"])}, "callbacks": [usage]},
        )
    tokens = next(iter(usage.usage_metadata.values()), {})
    return {
        "question": q["question"], "db_id": q["db_id"], "gold": q["query"],
        "pred": state["sql"], "status": state["status"], "attempts": state["attempts"],
        "error": state["errors"][-1] if state["errors"] else None,
        "category": state.get("error_category") or None,
        "seconds": round(time.monotonic() - started, 1),
        "input_tokens": tokens.get("input_tokens", 0), "output_tokens": tokens.get("output_tokens", 0),
    }

results = await asyncio.gather(*(run_one(q) for q in sample))
print(Counter(r["status"] for r in results), "| repairs:", sum(r["attempts"] for r in results))

Counter({'ok': 20}) | repairs: 1


## A rough check against gold

Run the gold SQL too and compare the rows, ignoring order. This is only for reading failures; the real score comes from Spider's own scorer in block 5.

In [4]:
def rows_of(out):
    return Counter(tuple(r.values()) for r in out["result"]) if not out["errors"] else None

for r in results:
    cfg = {"configurable": {"db_path": db_path(r["db_id"])}}
    gold_rows = rows_of(await execute_sql({"sql": r["gold"]}, cfg))
    pred_rows = rows_of(await execute_sql({"sql": r["pred"]}, cfg)) if r["pred"] else None
    r["rough_match"] = pred_rows is not None and pred_rows == gold_rows

print(f"rough match: {sum(r['rough_match'] for r in results)}/20")

rough match: 16/20


In [5]:
# read every failure: wrong rows, a failed run, or a repair along the way
for r in results:
    if r["rough_match"] and r["attempts"] == 0:
        continue
    print(f"[{r['db_id']}] {r['question']}")
    print(f"  status={r['status']} attempts={r['attempts']} category={r['category']} match={r['rough_match']}")
    print(f"  gold: {r['gold']}")
    print(f"  pred: {' '.join(r['pred'].split())}")
    if r["error"]:
        print(f"  error: {r['error'][:200]}")
    print()

# the validator's one blind spot: double quotes, which its Postgres parser reads as column names
print("predictions with double quotes:", [r["question"] for r in results if '"' in r["pred"]])

[world_1] What are the name, independence year, and surface area of the country with the smallest population?
  status=ok attempts=0 category=None match=False
  gold: SELECT Name ,  SurfaceArea ,  IndepYear FROM country ORDER BY Population LIMIT 1
  pred: SELECT Name, IndepYear, SurfaceArea FROM country WHERE Population = (SELECT MIN(Population) FROM country);

[student_transcripts_tracking] For each semester, what is the name and id of the one with the most students registered?
  status=ok attempts=0 category=None match=False
  gold: SELECT T1.semester_name ,  T1.semester_id FROM Semesters AS T1 JOIN Student_Enrolment AS T2 ON T1.semester_id  =  T2.semester_id GROUP BY T1.semester_id ORDER BY count(*) DESC LIMIT 1
  pred: WITH semester_counts AS ( SELECT s.semester_id, s.semester_name, COUNT(DISTINCT se.student_id) AS student_count FROM Semesters s LEFT JOIN Student_Enrolment se ON s.semester_id = se.semester_id GROUP BY s.semester_id, s.semester_name ) SELECT semester_id, semester_na

In [6]:
# one prediction per line, newlines flattened, same order as gold_20.sql
(OUT / "pred_20.sql").write_text("".join(" ".join(r["pred"].split()) + "\n" for r in results), encoding="utf-8")
(OUT / "run_20.json").write_text(json.dumps(results, indent=1, default=str), encoding="utf-8")
print("wrote", OUT / "pred_20.sql")

wrote c:\Users\HP\Desktop\prodsss\Text2SQL\data\eval\pred_20.sql


## What the full run will cost

Measured tokens from these 20, priced at `deepseek-v4-flash-0731` ($0.021 / M input, $0.32 / M output), scaled to 1,034 questions. The no-repair run makes exactly one call per question.

In [7]:
IN_PRICE, OUT_PRICE = 0.021 / 1e6, 0.32 / 1e6
calls = sum(1 + r["attempts"] for r in results)
tin, tout = sum(r["input_tokens"] for r in results), sum(r["output_tokens"] for r in results)
spent = tin * IN_PRICE + tout * OUT_PRICE
per_call = spent / calls
repair_rate = sum(r["attempts"] for r in results) / len(results)

print(f"this run: {calls} calls, {tin} in / {tout} out tokens, ${spent:.5f}")
print(f"per call: {tin/calls:.0f} in / {tout/calls:.0f} out tokens, ${per_call:.6f}")
print(f"repairs per question: {repair_rate:.2f}")
full = 1034 * per_call * ((1 + repair_rate) + 1)   # with-repair run + no-repair run
print(f"estimated full Spider cost (both runs): ${full:.3f}")

this run: 21 calls, 14656 in / 9191 out tokens, $0.00325
per call: 698 in / 438 out tokens, $0.000155
repairs per question: 0.05
estimated full Spider cost (both runs): $0.328
